In [ ]:
import os
import numpy as np
import networkx as nx
import geopandas as gpd
import matplotlib.pyplot as plt

In [ ]:
import sys

sys.path.append(os.path.abspath(".."))

In [ ]:
import src.city_data_processing.city_data_loader as cdl
import src.city_data_processing.city_graph_processing as cgp
import src.city_data_processing.city_blocks_processing as cbp
import src.city_data_processing.od_matrix_generator as odgen
import src.city_data_processing.pedestrian_graph_postprocessing as post
import src.utils.data_io as sl

import src.ga.LinePool as lp
import src.ga.TNDP as TNDP
import src.ga.GA_basic as GA

In [ ]:
from pathlib import Path

CITY_NAME = 'ust-ilimsk'
OSM_ID = 2380462

BASE_DIR = Path().resolve().parent
DATA_DIR = BASE_DIR / "data" / "cities" / CITY_NAME
GRAPHS_DIR = DATA_DIR / 'graphs/'

In [ ]:
from src.utils.plotters import CityPlotter

plotter = CityPlotter()

# Загрузка данных о территории

### Граница территории

In [ ]:
# polygon = cdl.get_bounary_from_file(boundary_filename)
polygon = cdl.get_boundary_from_osm(OSM_ID)

boundary_gdf = gpd.GeoDataFrame(geometry=[polygon], crs="EPSG:4326")
boundary_gdf = boundary_gdf.to_crs(boundary_gdf.estimate_utm_crs())
boundary_gdf["geometry"] = boundary_gdf.buffer(50)
boundary_gdf = boundary_gdf.to_crs(epsg=4326)

polygon_buffered = boundary_gdf.geometry[0]

### Кварталы

In [ ]:
blocks_filename = DATA_DIR / 'blocks_cut.geojson'
cut_blocks = gpd.read_file(blocks_filename)
cut_blocks.plot(linewidth=1, edgecolor='black')

## Загрузка графов

In [ ]:
if not os.path.isdir(GRAPHS_DIR):
    print('Creating directory for graphs')
    os.makedirs(GRAPHS_DIR, exist_ok=True)

In [ ]:
BLOCKS_GRAPH_FILEPATH = os.path.join(GRAPHS_DIR, "graph_blocks.pkl")
ROUTING_GRAPH_FILEPATH = os.path.join(GRAPHS_DIR, "graph_routing.pkl")

Графы УДС и пешехолдных путей загружаются в случае, если нет уже сформированных упрощенных графов

In [ ]:
roads_osm_tags = ['secondary', 'tertiary', 'unclassified']


def get_simplified_and_routing_graphs():
    DRIVE_GRAPH_FILEPATH = os.path.join(GRAPHS_DIR, "graph_drive.pkl")
    WALK_GRAPH_FILEPATH = os.path.join(GRAPHS_DIR, "graph_walk.pkl")

    if os.path.isfile(DRIVE_GRAPH_FILEPATH):
        print("Loading drive graph from file")
        G_drive_neat = sl.load_graph(DRIVE_GRAPH_FILEPATH)
    else:
        print("Loading drive graph from OSM")
        G_drive = cdl.get_streets_graph(
            polygon_buffered, 'drive', keep_largest_subgraph=True, osm_tags=roads_osm_tags)
        G_drive_neat = cgp.simplify_graph(G_drive, 200)
        sl.save_graph(G_drive_neat, DRIVE_GRAPH_FILEPATH)
    print("Drive graph loaded")

    if os.path.isfile(WALK_GRAPH_FILEPATH):
        print("Loading drive graph from file")
        G_walk_neat = sl.load_graph(WALK_GRAPH_FILEPATH)
    else:
        print("Loading walk graph from OSM")
        G_walk = cdl.get_streets_graph(polygon, 'walk')
        G_walk_neat = cgp.simplify_graph(G_walk, 5)
        sl.save_graph(G_walk_neat, WALK_GRAPH_FILEPATH)
    print("Walk graph loaded")

    buildings = cdl.get_buildings_from_OSM(polygon_buffered)

    G_simple = cbp.get_blocks_graph(
        G_walk_neat, cut_blocks, G_drive_neat, buildings)
    post.postprocess_graph(G_simple)
    sl.save_graph(G_simple, BLOCKS_GRAPH_FILEPATH)

    nodes_on_street = [
        n for n, data in G_simple.nodes(data=True)
        if data.get('on_street') == True
    ]
    G_routing = G_simple.subgraph(nodes_on_street).copy()
    post.postprocess_graph(G_routing, pedestrian=False)
    sl.save_graph(G_routing, ROUTING_GRAPH_FILEPATH)

    return G_simple, G_routing

Загрузка упрощенных графов

In [ ]:
if os.path.isfile(BLOCKS_GRAPH_FILEPATH):
    print("Loading blocks graph from file")
    G_blocks = sl.load_graph(BLOCKS_GRAPH_FILEPATH)

    if os.path.isfile(ROUTING_GRAPH_FILEPATH):
        print("Loading blocks graph from file")
        G_routing = sl.load_graph(ROUTING_GRAPH_FILEPATH)
    else:
        nodes_on_street = [
            n for n, data in G_blocks.nodes(data=True)
            if data.get('on_street') == True
        ]
        G_routing = G_blocks.subgraph(nodes_on_street).copy()
        post.postprocess_graph(G_routing, pedestrian=False)
        sl.save_graph(G_routing, ROUTING_GRAPH_FILEPATH)
else:
    G_blocks, G_routing = get_simplified_and_routing_graphs()

In [ ]:
ax = cut_blocks.to_crs(3857).plot(facecolor='None', edgecolor='black', linewidth=0.5, figsize=(10,10))
plotter.plot_streets_graph(G_blocks, ax=ax)

In [ ]:
plotter.plot_streets_graph(G_routing)

## Загрузка/генерация матрицы спроса

In [ ]:
population = 77000

In [ ]:
OD_FILEPATH = os.path.join(DATA_DIR, "od_modified_demand.npy")
BUILDINGS_FILEPATH = os.path.join(DATA_DIR, "buildings.geojson")
SERVICES_FILEPATH = os.path.join(DATA_DIR, "services.gpkg")

if os.path.isfile(OD_FILEPATH):
    print("Loading OD from file")
    od = sl.load_od_matrix(OD_FILEPATH)
else:
    print("Generating OD")
    buildings_osm = cdl.get_buildings_from_OSM(polygon).reset_index()
    buildings_osm['id'] = buildings_osm['id'].astype(str)
    
    if os.path.isfile(BUILDINGS_FILEPATH):
        buildings_manual = cdl.get_buildings_from_file(BUILDINGS_FILEPATH)
        buildings_manual['osm_id'] = buildings_manual['osm_id'].astype(str)
        buildings_osm = buildings_osm.drop(columns=['number_of_floors'], errors='ignore')
        
        buildings = buildings_osm.merge(
            buildings_manual[['osm_id', 'number_of_floors']],
            left_on='id',
            right_on='osm_id',
            how='left'
        )
    else:
        buildings = buildings_osm
    
    if os.path.isfile(SERVICES_FILEPATH):
        print("Loading services from file")
        services = cdl.get_services_from_file(SERVICES_FILEPATH)
    else:
        print("Loading services from OSM")
        services = cdl.get_services_from_osm(polygon_buffered)
        services.to_file(SERVICES_FILEPATH, layer="services", driver="GPKG")
    
    services_gdf = odgen.assign_services_capacity(services.to_crs(32645).copy())
    buildings_populated = odgen.assign_buildings_population(buildings.to_crs(32645).copy(), population)
    blocks_populated = odgen.assign_blocks_population(cut_blocks.to_crs(32645).copy(), buildings_populated)
        
    services_processed = odgen.assign_blocks_to_services(services_gdf, blocks_populated)
    od = odgen.generate_od_matrix_ipf(blocks_populated, services_processed, alpha=2)
    od.index = od.index.astype(str) + '_block'
    od.columns = od.columns.astype(str) + '_block'
    sl.save_od_matrix(od, OD_FILEPATH)

Визуализация спроса по кварталам

In [ ]:
cut_blocks['block_id'] = cut_blocks.index
cut_blocks['block_name'] = cut_blocks.index.astype(str) + "_block"

demand_per_block = od.sum(axis=1)  # сумма по строкам
cut_blocks['demand'] = cut_blocks['block_name'].map(demand_per_block)

fig, ax = plt.subplots(1, 1, figsize=(10, 10))

cut_blocks.plot(column='demand',  
                cmap='OrRd',      
                legend=True,     
                ax=ax,
                edgecolor='black',
                linewidth=0.5)

ax.axis('off')
plt.show()

## Загрузка/генерация пула маршрутов

In [ ]:
LINE_POOL_FILEPATH = os.path.join(DATA_DIR, "line_pool.json")

if os.path.isfile(LINE_POOL_FILEPATH):
    print("Loading line pool from file")
    line_pool = sl.load_line_pool(LINE_POOL_FILEPATH)
else:
    print("Generating line pool")
    connector_od = np.sqrt(odgen.generate_connector_od_matrix(G_routing, od))
    line_pool = lp.get_line_pool_connectors_only(nx.Graph(G_routing),
                                        connector_od,
                                        2,
                                        1000,
                                        10000,
                                        0.8,
                                        2)
    sl.save_line_pool(line_pool, LINE_POOL_FILEPATH)
    

# Генетический алгоритм

In [ ]:
tndp = TNDP.TNDP(graph=nx.Graph(G_routing),
                 pedestrian_graph=nx.Graph(G_blocks),
                 od_matrix=od,
                 line_pool=line_pool,
                 max_network_size=20,
                 time_weight=1,
                 cost_weight=10,
                 connectivity_weight=50000000)

ga = GA.GeneticAlgorithm(tndp, 
                         population_size=5,
                         initial_network_size=5,
                         n_generations=1)

In [ ]:
solution, fitness, gen = ga.generate_solution()

In [ ]:
solution

In [ ]:
plotter.plot_network_with_offset(solution, base_graph=G_routing, offset_step=20)

Сохранение результата работы

In [ ]:
SOLUTIONS_DIR = os.path.join(BASE_DIR / 'results' / 'solutions' / CITY_NAME)

In [ ]:
sl.save_experiment(solution, tndp, ga, SOLUTIONS_DIR + '/' + 'test.json')